# 协方差矩阵估计方法 - 真实数据回测

本notebook使用tushare获取真实市场数据进行回测

In [ ]:
import sys
sys.path.append('..')

import pandas as pd
import numpy as np
import matplotlib.pyplot as plt
import warnings
warnings.filterwarnings('ignore')

%matplotlib inline
plt.style.use('seaborn-v0_8-whitegrid')

from source.data_fetcher import DataFetcher
from source.covariance_estimators import CovarianceEstimator
from source.evaluation import PortfolioEvaluator
from source.portfolio_builders import PortfolioBuilder
from source.backtest import BacktestEngine
from source.strategies import BlackLittermanStrategy, RiskParityStrategy, BLMultiMethodBacktest, RiskParityMultiMethodBacktest

## 1. 初始化数据获取器

In [ ]:
data_fetcher = DataFetcher()
print("数据获取器已初始化，使用tushare pro API")

## 2. 定义要获取的资产配置

In [ ]:
asset_config = {
    '沪深300': '000300.SH',
    '标普500': 'SPX.GI',
    '恒生指数': 'HSI.HK',
    '中债-国债总财富': 'CBA00101.CI',
    '中债-企业债总财富': 'CBA00201.CI',
    '南华商品指数': 'NH0100.NH',
}

start_date = '20170101'
end_date = '20231231'

## 3. 获取数据（真实数据）

In [ ]:
print("尝试从tushare获取真实市场数据...")

try:
    returns_df = data_fetcher.fetch_asset_returns(asset_config, start_date, end_date)
    if len(returns_df) > 500:
        print(f"成功获取真实数据: {returns_df.shape}")
        print(returns_df.head())
    else:
        print("数据长度不足，将使用模拟数据")
        raise ValueError("数据不足")
except Exception as e:
    print(f"获取数据失败: {e}")
    print("使用模拟数据进行演示")
    
    np.random.seed(42)
    n_days = 1500
    n_assets = 6
    dates = pd.date_range(start='2017-01-01', periods=n_days, freq='B')
    
    base_returns = np.random.randn(n_days, n_assets) * 0.015
    correlation_matrix = np.array([
        [1.0, 0.5, 0.4, -0.1, -0.1, 0.3],
        [0.5, 1.0, 0.5, -0.1, -0.1, 0.4],
        [0.4, 0.5, 1.0, -0.1, -0.1, 0.3],
        [-0.1, -0.1, -0.1, 1.0, 0.7, 0.1],
        [-0.1, -0.1, -0.1, 0.7, 1.0, 0.1],
        [0.3, 0.4, 0.3, 0.1, 0.1, 1.0]
    ])
    
    L = np.linalg.cholesky(correlation_matrix)
    correlated_returns = base_returns @ L.T
    
    asset_names = list(asset_config.keys())
    returns_df = pd.DataFrame(correlated_returns, index=dates, columns=asset_names)
    print(f"模拟数据: {returns_df.shape}")

## 4. 初始化估计器和构建器

In [ ]:
cov_estimator = CovarianceEstimator()
portfolio_builder = PortfolioBuilder()
evaluator = PortfolioEvaluator()
backtest_engine = BacktestEngine(initial_capital=1000000)

print("所有模块初始化完成")

## 5. 运行回测

In [ ]:
methods = [
    'sample_cov',
    'ledoit_wolf_constant_variance',
    'ledoit_wolf_single_factor',
    'ledoit_wolf_constant_correlation',
    'risk_metrics',
]

print("开始最低波动组合回测...")
min_var_results = {}

for method in methods:
    print(f"  回测: {method}")
    result = backtest_engine.run_rolling_backtest(
        returns=returns_df,
        cov_estimator=cov_estimator,
        portfolio_builder=portfolio_builder,
        method=method,
        lookback_period=252,
        rebalance_freq='monthly',
        allow_short=False,
        portfolio_type='min_variance'
    )
    min_var_results[method] = result

print("\n最低波动组合回测完成!")

## 6. 性能对比

In [ ]:
comparison_df = backtest_engine.compare_methods()
print("最低波动组合性能对比:")
print(comparison_df.round(4))

In [ ]:
fig, ax = plt.subplots(figsize=(12, 6))

for method, result in min_var_results.items():
    if result is not None and len(result) > 0:
        portfolio_values = result['portfolio_value']
        if isinstance(portfolio_values, pd.Series):
            ax.plot(portfolio_values.values, label=method, linewidth=1.5)

ax.set_title('最低波动组合 - 组合价值曲线', fontsize=14)
ax.set_xlabel('时间')
ax.set_ylabel('组合价值')
ax.legend(loc='best', fontsize=9)
ax.grid(True, alpha=0.3)
plt.tight_layout()
plt.savefig('../output/min_variance_real_data.png', dpi=150, bbox_inches='tight')
plt.show()
print("图表已保存")

## 7. Black-Litterman策略回测

In [ ]:
bl_backtest = BLMultiMethodBacktest(initial_capital=1000000)
market_cap_weights = np.array([0.15, 0.20, 0.10, 0.35, 0.15, 0.05])

bl_methods = ['sample_cov', 'ledoit_wolf_single_factor', 'risk_metrics']

bl_results = bl_backtest.run_backtest(
    returns=returns_df,
    cov_estimator=cov_estimator,
    market_cap_weights=market_cap_weights,
    methods=bl_methods,
    lookback_period=252*5,
    rebalance_freq='monthly',
    allow_short=False
)

print("Black-Litterman策略回测完成")

In [ ]:
bl_metrics = []
for method, result in bl_results.items():
    daily_returns = result['daily_returns']
    if len(daily_returns) > 0:
        ann_return = (1 + daily_returns.mean()) ** 252 - 1
        ann_vol = daily_returns.std() * np.sqrt(252)
        sharpe = ann_return / ann_vol if ann_vol > 0 else 0
        bl_metrics.append({
            'Method': method,
            'Annualized Return': ann_return,
            'Annualized Volatility': ann_vol,
            'Sharpe Ratio': sharpe
        })

bl_metrics_df = pd.DataFrame(bl_metrics).set_index('Method')
print("Black-Litterman策略性能:")
print(bl_metrics_df.round(4))

## 8. 风险平价策略回测

In [ ]:
rp_backtest = RiskParityMultiMethodBacktest(initial_capital=1000000)

rp_results = rp_backtest.run_backtest(
    returns=returns_df,
    cov_estimator=cov_estimator,
    methods=bl_methods,
    lookback_period=126,
    rebalance_freq='monthly',
    allow_short=False
)

print("风险平价策略回测完成")

In [ ]:
rp_metrics = []
for method, result in rp_results.items():
    daily_returns = result['daily_returns']
    if len(daily_returns) > 0:
        ann_return = (1 + daily_returns.mean()) ** 252 - 1
        ann_vol = daily_returns.std() * np.sqrt(252)
        sharpe = ann_return / ann_vol if ann_vol > 0 else 0
        rp_metrics.append({
            'Method': method,
            'Annualized Return': ann_return,
            'Annualized Volatility': ann_vol,
            'Sharpe Ratio': sharpe
        })

rp_metrics_df = pd.DataFrame(rp_metrics).set_index('Method')
print("风险平价策略性能:")
print(rp_metrics_df.round(4))

In [ ]:
fig, axes = plt.subplots(1, 2, figsize=(16, 6))

ax1 = axes[0]
for method, result in bl_results.items():
    portfolio_values = result['portfolio_values']
    ax1.plot(portfolio_values.values, label=method, linewidth=1.5)
ax1.set_title('Black-Litterman策略', fontsize=14)
ax1.legend(loc='best')
ax1.grid(True, alpha=0.3)

ax2 = axes[1]
for method, result in rp_results.items():
    portfolio_values = result['portfolio_values']
    ax2.plot(portfolio_values.values, label=method, linewidth=1.5)
ax2.set_title('风险平价策略', fontsize=14)
ax2.legend(loc='best')
ax2.grid(True, alpha=0.3)

plt.tight_layout()
plt.savefig('../output/strategies_comparison.png', dpi=150, bbox_inches='tight')
plt.show()
print("策略对比图已保存")